## Points to improve
- ~smoothen slope using 5th and 95th percentiles~
- ~convert decibels to linear~
- ~test code in Kashmir (valley)~
- push low level code to utils and other relevant modules
- final checks on documentation

In [1]:
import sys
sys.path.append('../')

from ifmiap import flood_mapper
from ifmiap import utils
import geopandas as gpd
import time

In [2]:
# small piece of code to get zone-wise list of IDs
gdf = gpd.read_file(r'../resources/india_utm_fishnet_buffer.gpkg')
zone_id_group = gdf[['zone', 'ID']].groupby('zone')['ID'].apply(list)

zone_id_dict = dict()

for idx in zone_id_group.index:
    zone_id_dict[idx] = zone_id_group[idx]

In [3]:
%%time

for ID in [321]:
    t1 = time.time()
    
    print(f'Processing ID: {ID}')
    # create the flood mapper class
    flood_mapper_obj = flood_mapper(
        grid_shapefile = r'../resources/india_utm_fishnet_buffer.gpkg',
        grid_id_list = [ID],#zone_id_dict['45R'],#[378, 384, 390],
        dry_date_col = 'dry_month',
        id_col = 'ID',
        dry_years=[2020, 2020],
        slope_dir = r'../resources/slope/',
        wet_duration = ['2020/07', '2020/07']
    )

    flood_mapper_obj.get_dry_dates()

    if len(flood_mapper_obj.aoi_ids_to_process) > 0:
        flood_mapper_obj.generate_dry_date_ranges()
        flood_mapper_obj.get_s1_items(dry_wet='dry')
        flood_mapper_obj.read_scenes(dry_wet='dry', overview_level=2)
        flood_mapper_obj.generate_mean_std_by_aoi()
    else:
        flood_mapper_obj.load_mean_std_by_aoi()

    flood_mapper_obj.prepare_slope(dem_overview=0, buffer=500)
    flood_mapper_obj.prepare_wet_scenes(overview_level=2)
    flood_mapper_obj.generate_number_of_scenes(export_raster=True)
    flood_mapper_obj.map_floods(vv_thd=-2.5, vh_thd=-2.5, rel_slope_thd=20,
                                  export_raster=False, export_vector=True, export_maps=False)
    flood_mapper_obj.merge_floods_by_date(export_raster=True)
    flood_mapper_obj.monthly_sum()
    
    t2 = time.time()
    t_delta = t2 - t1
    print(f'Total time taken: {(t_delta / 60):.2f} mins.')

Processing ID: 321


/usr/local/lib/python3.8/dist-packages/pystac_client/item_search.py:835: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/rioxarray/raster_writer.py:132: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/rioxarray/raster_writer.py:132: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/rioxarray/raster_writer.py:132: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/rioxarray/raster_writer.py:132: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.40282346638

Slope for tile ID 321 not found. Downloading DEM...


/usr/local/lib/python3.8/dist-packages/pystac_client/item_search.py:835: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/rioxarray/raster_writer.py:132: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/rioxarray/raster_writer.py:132: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/rioxarray/raster_writer.py:132: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.4028234663852886e+38) to match the dtype of the data.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/rioxarray/raster_writer.py:132: UserWarning: The nodata value (3.402823466e+38) has been automatically changed to (3.40282346638

Total time taken: 5.18 mins.
CPU times: user 1min 35s, sys: 5.36 s, total: 1min 40s
Wall time: 5min 11s
